# PR Opportunity Monitor - Multi-Client
### MIS285N Generative AI I - Prof. Caramanis

**What this system does:** Given a corpus of published articles for each PR client, automatically surface relevant incoming news stories and generate grounded pitch drafts.

**Course concepts applied:**
- **Bi-encoder (BERT)** - dense embeddings via `all-MiniLM-L6-v2` (sentence-transformers / HuggingFace)
- **TF-IDF** - sparse bag-of-words retrieval as a complementary signal
- **Hybrid search** - `score = alpha x dense + (1 - alpha) x sparse` with alpha tuned per client
- **ChromaDB** - vector store with one collection per client (prevents cross-client contamination)
- **Recall@K** - leave-one-out evaluation; uses article title as query where available, falls back to first 300 chars
- **RAG** - retrieve top-K corpus articles as context, pass to LLM to generate grounded pitch
- **Human feedback loop** - captures expert ratings for future reranking improvement

**Clients:** Client 1 (personal injury law) - Client 2 (healthcare) - Client 3 (telecom) - Client 4 (business law)

**Design decisions documented in comments throughout.**

In [4]:
# Cell 1 - Install packages
!pip install -q sentence-transformers chromadb scikit-learn groq newsapi-python matplotlib python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

In [5]:
# Cell 2 - Imports and API keys
import os, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import chromadb

from datetime import datetime, timedelta
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown

try:
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'NEWSAPI_KEY']:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            print(f'{key} loaded')
except Exception:
    print('No Colab secrets found - template pitches will be used, demo mode for news')

print('Imports ready')

No Colab secrets found - template pitches will be used, demo mode for news
Imports ready


In [6]:
# Cell 3 - Configuration
#
# THRESHOLDS
# ALPHA GRID alpha=1.0 wins for all clients
#
# CORPUS_STOPWORDS - 'arizona' and city names added to TF-IDF stopwords
#   Every corpus article and every news headline mentions Arizona.
#   The token carries zero discriminating information - it appears
#   everywhere and therefore must be removed or it artificially inflates
#   TF-IDF scores for any headline that simply mentions the state.

STRONG_THRESHOLD   = 0.55
ADJACENT_THRESHOLD = 0.40

DEFAULT_ALPHA = 1.0 #found as best parameters from the test in cell 6
DEFAULT_K     = 3
NEWS_LOOKBACK_DAYS = 7

CORPUS_STOPWORDS = ['arizona', 'phoenix', 'az', 'tucson', 'scottsdale',
                    'tempe', 'mesa', 'chandler', 'gilbert', 'glendale']

BLOCKED_DOMAINS = {
    'globenewswire.com', 'prnewswire.com', 'businesswire.com',
    'newswire.com', 'einpresswire.com', 'accesswire.com', 'prweb.com'
}

def is_blocked(url):
    return any(d in (url or '').lower() for d in BLOCKED_DOMAINS)

CLIENTS = {
    'client1':      {'name': 'Client 1',        'file': 'client1_data.csv',      'alpha': DEFAULT_ALPHA, 'k': DEFAULT_K},
    'client2':       {'name': 'Client 2',  'file': 'client2_data.csv',       'alpha': DEFAULT_ALPHA, 'k': DEFAULT_K},
    'client3': {'name': 'Client 3',      'file': 'client3_data.csv', 'alpha': DEFAULT_ALPHA, 'k': DEFAULT_K},
    'client4':     {'name': 'Client 4',        'file': 'client4_data.csv',     'alpha': DEFAULT_ALPHA, 'k': DEFAULT_K},
}

# News queries per client
# topic-specific queries, no bare 'arizona'
#   Previous version used 'arizona' as a catch-all which pulled hundreds of
#   sports/entertainment headlines. These scored 0.30-0.47 purely because
#   'arizona' appeared in both headline and corpus. Replaced with
#   topic-specific queries so the input pool is domain-relevant before
#   the matcher runs. The semantic matcher still does the real filtering.
CLIENT_NEWS_QUERIES = {
    'client1': [
        'arizona car accident crash',
        'arizona personal injury lawsuit',
        'arizona DUI drunk driving',
        'arizona distracted driving texting',
        'arizona traffic law',
        'arizona dog bite',
        'arizona pedestrian safety',
        'arizona wrongful death',
        'arizona pool drowning',
        'arizona marijuana driving',
    ],
    'client2': [
        'arizona health care',
        'arizona diabetes obesity',
        'arizona mental health',
        'arizona medicare medicaid',
        'arizona primary care',
        'health awareness month',
        'arizona public health',
        'arizona senior care',
    ],
   'client3': [
        'Cox Communications',
        'arizona broadband',
        'arizona internet',
        'arizona telecommunications',
],
    'client4': [
        'arizona business law',
        'arizona employment law',
        'arizona commercial real estate',
        'arizona corporate lawsuit',
        'arizona regulatory compliance',
        'phoenix business legal',
    ],
}

print('Config ready')
print(f'Clients: {[v["name"] for v in CLIENTS.values()]}')
print(f'Thresholds: strong>={STRONG_THRESHOLD}, adjacent>={ADJACENT_THRESHOLD}')

Config ready
Clients: ['Client 1', 'Client 2', 'Client 3', 'Client 4']
Thresholds: strong>=0.55, adjacent>=0.4


In [7]:
# Cell 4 - Upload training data
from google.colab import files
import re

print('Upload all 4 CSV files:')
print('  client1_data.csv')
print('  client2_data.csv')
print('  client3_data.csv')
print('  client4_data.csv')
uploaded = files.upload()
# Normalize filenames - strip Colab's " (2)", " (3)" etc suffixes
uploaded = {re.sub(r'\s*\(\d+\)(?=\.)', '', k): v for k, v in uploaded.items()}

client_data = {}
for client_id, cfg in CLIENTS.items():
    fname = cfg['file']
    if fname in uploaded:
        import io
        df = pd.read_csv(io.BytesIO(uploaded[fname]))
        # Handle full_text vs text column name difference
        if 'full_text' in df.columns and 'text' not in df.columns:
            df = df.rename(columns={'full_text': 'text'})
        df['text']   = df['text'].fillna('').astype(str)
        df['title']  = df['title'].fillna('').astype(str) if 'title' in df.columns else ''
        df['topic']  = df['topic'].fillna('').astype(str) if 'topic' in df.columns else ''
        df['outlet'] = df['outlet'].fillna('').astype(str) if 'outlet' in df.columns else ''
        df['date']   = df['date'].fillna('').astype(str) if 'date' in df.columns else ''
        df['url']    = df['url'].fillna('').astype(str) if 'url' in df.columns else ''
        client_data[client_id] = df
        has_titles = (df['title'] != '').sum()
        print(f"{cfg['name']}: {len(df)} articles, {has_titles} with titles")
    else:
        print(f'WARNING: {fname} not found in upload')

print(f'\nTotal corpus: {sum(len(df) for df in client_data.values())} articles')

Upload all 4 CSV files:
  client1_data.csv
  client2_data.csv
  client3_data.csv
  client4_data.csv


Saving client1_data.csv to client1_data.csv
Saving client2_data.csv to client2_data.csv
Saving client3_data.csv to client3_data.csv
Saving client4_data.csv to client4_data.csv
Client 1: 120 articles, 0 with titles
Client 2: 137 articles, 0 with titles
Client 3: 609 articles, 0 with titles
Client 4: 141 articles, 0 with titles

Total corpus: 1007 articles


In [8]:
# Cell 5 - Build vector stores (Dense + Sparse)
#
# SPARSE - TF-IDF stopwords include CORPUS_STOPWORDS:
#   'arizona' and city names are added on top of standard English stopwords.
#   Without this, 'arizona' dominates TF-IDF scores since it appears in
#   every document and every headline - making it useless as a signal
#   and causing any Arizona-keyword headline to score artificially high.

# Clear existing collections to avoid duplicate error on re-run
for client_id in CLIENTS:
    try:
        chroma_client.delete_collection(f'pr_monitor_{client_id}')
    except:
        pass

print('Loading bi-encoder model (all-MiniLM-L6-v2)...')
model         = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()

dense_collections  = {}
sparse_vectorizers = {}
sparse_matrices    = {}

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
combined_stopwords = list(ENGLISH_STOP_WORDS) + CORPUS_STOPWORDS

for client_id, df in client_data.items():
    name = CLIENTS[client_id]['name']
    print(f'\nBuilding vector store: {name} ({len(df)} articles)...')

    collection = chroma_client.create_collection(
        name=f'pr_monitor_{client_id}',
        metadata={'hnsw:space': 'cosine'}
    )
    embeddings = model.encode(df['text'].tolist(), normalize_embeddings=True, show_progress_bar=False)
    collection.add(
        ids=[str(i) for i in range(len(df))],
        embeddings=embeddings.tolist(),
        metadatas=[{
            'topic':       str(row.get('topic', '')),
            'outlet':      str(row.get('outlet', '')),
            'date':        str(row.get('date', '')),
            'url':         str(row.get('url', '')),
            'title':       str(row.get('title', '')),
            'is_calendar': str(row.get('outlet', '') == 'health_awareness_calendar'),
        } for _, row in df.iterrows()],
        documents=df['text'].tolist()
    )
    dense_collections[client_id] = collection

    tfidf  = TfidfVectorizer(max_features=10000, stop_words=combined_stopwords, ngram_range=(1,2))
    matrix = tfidf.fit_transform(df['text'].tolist())
    sparse_vectorizers[client_id] = tfidf
    sparse_matrices[client_id]    = matrix
    print(f'  Done - {len(df)} articles indexed')

print('\nAll vector stores ready.')

Loading bi-encoder model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Building vector store: Client 1 (120 articles)...
  Done - 120 articles indexed

Building vector store: Client 2 (137 articles)...
  Done - 137 articles indexed

Building vector store: Client 3 (609 articles)...
  Done - 609 articles indexed

Building vector store: Client 4 (141 articles)...
  Done - 141 articles indexed

All vector stores ready.


In [11]:
# # Cell 6 - Evaluation: Leave-One-Out Retrieval Consistency
# #
# # WHAT THIS MEASURES:
# #   For each article, its title is used as a query against the remaining corpus
# #   (the article itself is excluded). A hit is counted when at least one retrieved
# #   result scores >= STRONG_THRESHOLD (0.55). This measures whether the embedding
# #   space is internally consistent -- i.e., does a corpus article's title reliably
# #   surface semantically similar articles? This is not Recall@K in the strict IR
# #   sense (which would require labeled relevance judgments), but a valid proxy for
# #   retrieval quality given the absence of a labeled test set.
# #
# # WHY LEAVE-ONE-OUT (not 80/20):
# #   The bi-encoder is frozen -- no model weights are being trained.
# #   Leave-one-out is standard when evaluating a fixed retrieval index.
# #   With small corpora, holding out 20% would remove valuable coverage.
# #
# # WHY K IS REMOVED:
# #   The consistency metric checks whether any result clears the threshold,
# #   not whether a specific document appears at rank K. K does not affect
# #   the outcome so the grid tests alpha only.
# #
# # QUERY CONSTRUCTION - title preferred over article opening:
# #   Article openings share boilerplate with the corpus, artificially inflating
# #   TF-IDF scores and biasing results toward lower alpha. Titles are short and
# #   headline-like, which is more faithful to production inputs.
# #
# # ALPHA GRID - extended to 1.0:
# #   Previous grid peaked at 0.9 -- never confirmed a true optimum. Extended
# #   grid either finds the real peak or confirms alpha=1.0 (pure dense) is best.

# # don't need to run every time -- best parameters are hardcoded in Cell 3.
# # uncomment and run when onboarding a new client or changing STRONG_THRESHOLD.

# def hybrid_search_eval(query, client_id, exclude_idx, alpha):
#     df      = client_data[client_id]
#     q_emb   = model.encode([query], normalize_embeddings=True)
#     results = dense_collections[client_id].query(
#         query_embeddings=q_emb.tolist(),
#         n_results=min(10, len(df))
#     )
#     dense_ids    = [int(i) for i in results['ids'][0]]
#     dense_scores = [1 - d for d in results['distances'][0]]
#     dense_map    = dict(zip(dense_ids, dense_scores))
#     q_tfidf       = sparse_vectorizers[client_id].transform([query])
#     sparse_scores = cosine_similarity(q_tfidf, sparse_matrices[client_id]).flatten()
#     scored = []
#     for idx in dense_ids:
#         if idx == exclude_idx:
#             continue
#         score = alpha * dense_map.get(idx, 0.0) + (1 - alpha) * float(sparse_scores[idx])
#         scored.append((idx, score))
#     scored.sort(key=lambda x: x[1], reverse=True)
#     return scored

# def retrieval_consistency(client_id, alpha):
#     """
#     Leave-one-out retrieval consistency score.
#     For each article, uses its title as a query against the remaining corpus.
#     Counts a hit when at least one result scores >= STRONG_THRESHOLD.
#     Measures whether the embedding space is internally consistent --
#     not held-out generalization in the strict IR sense.
#     """
#     df   = client_data[client_id]
#     hits = 0
#     for idx in range(len(df)):
#         row   = df.iloc[idx]
#         title = str(row.get('title', '')).strip()
#         query = title if title else str(row['text'])[:300]
#         scored = hybrid_search_eval(query, client_id, exclude_idx=idx, alpha=alpha)
#         if any(score >= STRONG_THRESHOLD for _, score in scored):
#             hits += 1
#     return hits / len(df) if len(df) > 0 else 0.0

# ALPHAS = [0.3, 0.5, 0.7, 0.9, 0.95, 1.0]

# print('Running leave-one-out retrieval consistency evaluation...')
# print('Query: article title where available, else first 300 chars')
# print('Hit = at least one retrieved result scores >= STRONG_THRESHOLD')
# print('(This takes a few minutes)\n')

# eval_results = {}
# for client_id, cfg in CLIENTS.items():
#     print(f"--- {cfg['name']} ---")
#     best_score, best_alpha = 0, DEFAULT_ALPHA
#     rows = []
#     for alpha in ALPHAS:
#         score = retrieval_consistency(client_id, alpha)
#         rows.append({'alpha': alpha, 'consistency': round(score, 3)})
#         print(f'  alpha={alpha}  consistency={score:.3f}')
#         if score > best_score:
#             best_score, best_alpha = score, alpha
#     CLIENTS[client_id]['alpha'] = best_alpha
#     eval_results[client_id]     = rows
#     print(f'  BEST: alpha={best_alpha}, consistency={best_score:.3f}\n')

# print('Evaluation complete. Best alpha saved per client.')
# print('Note: if alpha=1.0 wins for all clients, TF-IDF adds no value - pure dense is sufficient.')

Running leave-one-out retrieval consistency evaluation...
Query: article title where available, else first 300 chars
Hit = at least one retrieved result scores >= STRONG_THRESHOLD
(This takes a few minutes)

--- Client 1 ---
  alpha=0.3  consistency=0.100
  alpha=0.5  consistency=0.283
  alpha=0.7  consistency=0.508
  alpha=0.9  consistency=0.783
  alpha=0.95  consistency=0.850
  alpha=1.0  consistency=0.875
  BEST: alpha=1.0, consistency=0.875

--- Client 2 ---
  alpha=0.3  consistency=0.139
  alpha=0.5  consistency=0.285
  alpha=0.7  consistency=0.555
  alpha=0.9  consistency=0.759
  alpha=0.95  consistency=0.796
  alpha=1.0  consistency=0.847
  BEST: alpha=1.0, consistency=0.847

--- Client 3 ---
  alpha=0.3  consistency=0.282
  alpha=0.5  consistency=0.417
  alpha=0.7  consistency=0.576
  alpha=0.9  consistency=0.732
  alpha=0.95  consistency=0.767
  alpha=1.0  consistency=0.798
  BEST: alpha=1.0, consistency=0.798

--- Client 4 ---
  alpha=0.3  consistency=0.071
  alpha=0.5  consi

In [12]:
# Cell 7 - Hybrid search

def hybrid_search(query, client_id, alpha=None, k=None):
    alpha = alpha or CLIENTS[client_id]['alpha']
    k     = k     or CLIENTS[client_id]['k']
    df    = client_data[client_id]
    q_emb   = model.encode([query], normalize_embeddings=True)
    results = dense_collections[client_id].query(
        query_embeddings=q_emb.tolist(),
        n_results=min(k * 2, len(df))
    )
    dense_ids    = [int(i) for i in results['ids'][0]]
    dense_scores = [1 - d for d in results['distances'][0]]
    dense_map    = dict(zip(dense_ids, dense_scores))
    q_tfidf       = sparse_vectorizers[client_id].transform([query])
    sparse_scores = cosine_similarity(q_tfidf, sparse_matrices[client_id]).flatten()
    scored = []
    for idx in dense_ids:
        d_score = dense_map.get(idx, 0.0)
        s_score = float(sparse_scores[idx])
        score   = alpha * d_score + (1 - alpha) * s_score
        row     = df.iloc[idx]
        outlet  = str(row.get('outlet', ''))
        outlet  = outlet if outlet and outlet != 'nan' else ''
        scored.append({
            'idx':          idx,
            'score':        round(score, 4),
            'dense_score':  round(d_score, 4),
            'sparse_score': round(s_score, 4),
            'topic':        str(row.get('topic', '')),
            'outlet':       outlet,
            'date':         str(row.get('date', '')),
            'url':          str(row.get('url', '')),
            'title':        str(row.get('title', '')),
            'text':         row['text'],
            'is_calendar':  row.get('outlet', '') == 'health_awareness_calendar',
        })
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:k]

def match_headline(headline, client_id):
    if is_blocked(headline):
        return {'tier': 'blocked', 'results': []}
    results = hybrid_search(headline, client_id)
    if not results:
        return {'tier': 'no_match', 'results': []}
    top_score = results[0]['score']
    if top_score >= STRONG_THRESHOLD:
        tier = 'strong'
    elif top_score >= ADJACENT_THRESHOLD:
        tier = 'adjacent'
    else:
        tier = 'no_match'
    return {'tier': tier, 'score': top_score, 'results': results}

print('Hybrid search ready.')
for cid, cfg in CLIENTS.items():
    print(f"  {cfg['name']}: alpha={cfg['alpha']}, K={cfg['k']}")

Hybrid search ready.
  Client 1: alpha=1.0, K=3
  Client 2: alpha=1.0, K=3
  Client 3: alpha=1.0, K=3
  Client 4: alpha=1.0, K=3


In [13]:

# Cell 8 - Pull live news headlines
#
# DESIGN DECISION - Google News RSS as primary source, NewsAPI as fallback:
#   NewsAPI free tier is limited to 100 requests/24 hours. With 4 clients
#   and topic-specific queries this quota is easily exhausted in a single run.
#   Google News RSS has no rate limits, no API key required, and provides
#   better local news coverage for Arizona-specific topics.
#   NewsAPI is used as fallback if NEWSAPI_KEY is set and RSS fails.
#
# RSS query strategy: same topic-specific queries as before, passed as
#   URL parameters to Google News RSS search endpoint.

!pip install -q feedparser

import feedparser, time, urllib.parse, re
from datetime import timezone

# DATE FILTERING
#   Two-layer approach to catch stale content:
#   1. published_parsed date check - filters articles older than NEWS_LOOKBACK_DAYS.
#      Entries with no parseable date are skipped entirely - undated RSS entries
#      are almost always old recycled content.
#   2. Old-year title pattern - catches articles Google re-surfaces with a fresh
#      timestamp (e.g. "Self-Driving Uber Kills Pedestrian (Published 2018)").
#      Google re-timestamps old articles when they get re-shared, so the date
#      check alone won't catch them - we also scan the title for the year tag.

CUTOFF          = datetime.now(timezone.utc) - timedelta(days=NEWS_LOOKBACK_DAYS)
OLD_YEAR_PATTERN = re.compile(r'\(Published (19\d\d|20[01]\d|202[0-3])\)')

def fetch_rss_headlines(query, max_results=50):
    """Fetch headlines from Google News RSS for a given query."""
    encoded = urllib.parse.quote(query)
    url     = f'https://news.google.com/rss/search?q={encoded}&hl=en-US&gl=US&ceid=US:en'
    try:
        feed  = feedparser.parse(url)
        items = []
        for entry in feed.entries[:max_results]:

            # LAYER 1 - date filter
            #   published_parsed is feedparser's pre-parsed time tuple, more reliable
            #   than parsing the raw published string which varies by feed.
            #   Skip entries with no date - better to miss than surface stale content.
            published = entry.get('published_parsed')
            if not published:
                continue
            pub_dt = datetime(*published[:6], tzinfo=timezone.utc)
            if pub_dt < CUTOFF:
                continue

            title = entry.get('title', '').strip()
            # Google News appends source like "Title - Source", strip it
            if ' - ' in title:
                title = title.rsplit(' - ', 1)[0].strip()
            if not title:
                continue

            # LAYER 2 - old-year title pattern
            #   Google re-timestamps old articles when re-shared so they pass
            #   the date check above. Catch them by scanning for the (Published YYYY)
            #   tag Google appends to archived articles.
            if OLD_YEAR_PATTERN.search(title):
                continue

            days_ago = (datetime.now(timezone.utc) - pub_dt).days
            items.append({
                'title':       title,
                'url':         entry.get('link', ''),
                'publishedAt': entry.get('published', ''),
                'days_ago':    days_ago,
            })
        return items
    except Exception as e:
        print(f'  RSS error ({query}): {e}')
        return []

all_headlines = {}

for client_id, cfg in CLIENTS.items():
    seen_titles = set()
    headlines   = []
    queries     = CLIENT_NEWS_QUERIES[client_id]

    for q in queries:
        results = fetch_rss_headlines(q, max_results=50)
        for h in results:
            if h['title'] not in seen_titles and not is_blocked(h['url']):
                seen_titles.add(h['title'])
                headlines.append(h)
        time.sleep(0.3)  # polite delay between requests

    headlines.sort(key=lambda x: x.get('publishedAt', ''), reverse=True)
    all_headlines[client_id] = headlines
    print(f"{cfg['name']}: {len(headlines)} unique headlines")

# Fallback to NewsAPI if RSS returned nothing and key is available
if all(len(h) == 0 for h in all_headlines.values()) and os.environ.get('NEWSAPI_KEY'):
    print('\nRSS returned nothing - falling back to NewsAPI...')
    from newsapi import NewsApiClient
    newsapi   = NewsApiClient(api_key=os.environ['NEWSAPI_KEY'])
    from_date = (datetime.now() - timedelta(days=NEWS_LOOKBACK_DAYS)).strftime('%Y-%m-%d')
    to_date   = datetime.now().strftime('%Y-%m-%d')
    for client_id, cfg in CLIENTS.items():
        seen_urls = set()
        headlines = []
        for q in CLIENT_NEWS_QUERIES[client_id]:
            try:
                resp = newsapi.get_everything(
                    q=q, language='en',
                    from_param=from_date, to=to_date,
                    sort_by='relevancy', page_size=100
                )
                for article in resp.get('articles', []):
                    url   = article.get('url', '')
                    title = article.get('title', '').strip()
                    if not title or is_blocked(url) or url in seen_urls or '[Removed]' in title:
                        continue
                    seen_urls.add(url)
                    headlines.append({
                        'title':       title,
                        'url':         url,
                        'publishedAt': article.get('publishedAt', ''),
                        'days_ago':    0,
                    })
            except Exception as e:
                print(f'  NewsAPI error ({q}): {e}')
        all_headlines[client_id] = headlines
        print(f"{cfg['name']}: {len(headlines)} headlines (NewsAPI)")

print('\nHeadlines ready.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.0 MB/s eta 0:00:00
Client 1: 21 unique headlines
Client 2: 29 unique headlines
Client 3: 7 unique headlines
Client 4: 14 unique headlines

Headlines ready.


In [14]:
# Cell 9 - Match headlines against each client corpus

all_matches = {}
for client_id, headlines in all_headlines.items():
    name     = CLIENTS[client_id]['name']
    strong   = []
    adjacent = []
    for h in headlines:
        title  = h['title']
        result = match_headline(title, client_id)
        if result['tier'] in ('strong', 'adjacent'):
            match = {
                'headline':    title,
                'url':         h.get('url', ''),
                'published':   h.get('publishedAt', ''),
                'days_ago':    h.get('days_ago', 0),
                'score':       result['score'],
                'tier':        result['tier'],
                'top_results': result['results'],
                'stale':       h.get('days_ago', 0) > 3,
            }
            if result['tier'] == 'strong':
                strong.append(match)
            else:
                adjacent.append(match)
    all_matches[client_id] = {'strong': strong, 'adjacent': adjacent}
    print(f"{name}: {len(strong)} strong, {len(adjacent)} adjacent, "
          f"{len(headlines) - len(strong) - len(adjacent)} no match")

print('\nMatching complete.')

Client 1: 4 strong, 9 adjacent, 8 no match
Client 2: 7 strong, 13 adjacent, 9 no match
Client 3: 4 strong, 2 adjacent, 1 no match
Client 4: 2 strong, 7 adjacent, 5 no match

Matching complete.


In [16]:
# Cell 10 - RAG Pitch Generator
#
# DESIGN DECISION - pitch prompt rewritten to explain topical connection:
#  A PR pitch needs to answer: what does this client know about THIS topic specifically?
#  instructs the LLM to:
#     1. Identify what subject matter expertise the retrieved articles show
#     2. Explain the connection between that expertise and this headline
#     3. Propose one concrete angle - not generic talking points
#
# GROQ MODEL - Using llama-3.3-70b-versatile.

GROQ_MODEL    = 'llama-3.3-70b-versatile'
current_month = datetime.now().strftime('%B')

def generate_pitch(headline, client_id, top_results):
    client_name = CLIENTS[client_id]['name']
    top         = top_results[0]
    is_calendar = top.get('is_calendar', False)

    # Build RAG context - include topic and title so LLM understands the expertise
    context_parts = []
    for i, r in enumerate(top_results, 1):
        if r['is_calendar']:
            source = f"[{client_name} 2026 Health Awareness Calendar - {r['topic']}]"
        else:
            title_str  = f' | \"{r["title"]}\"' if r.get('title') else ''
            outlet_str = r['outlet'] if r.get('outlet') else 'trade press'
            source     = f'[{outlet_str} {r["date"]}{title_str}]'
        context_parts.append(f"Article {i} {source}:\n{r['text'][:400]}")
    context = '\n\n'.join(context_parts)

    if os.environ.get('GROQ_API_KEY'):
        try:
            from groq import Groq
            groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])

            if is_calendar:
                framing = (f'This is a PROACTIVE pitch. {client_name} 2026 health awareness '
                           f'calendar focuses on {top["topic"]} this period.')
            else:
                framing = f'This is a REACTIVE pitch. {client_name} has directly relevant published expertise.'

            prompt = (
                f'You are a PR professional preparing a briefing for a client.\n\n'
                f'NEWS HEADLINE: {headline}\n\n'
                f'CLIENT: {client_name}\n\n'
                f'{framing}\n\n'
                f'RETRIEVED EXPERTISE:\n{context}\n\n'
                f'NOTE: The retrieved articles above are PAST published pieces used only to establish expertise. Do not cite any statistics, dates, or facts from them as if they are current.\n\n'
                f'Respond in exactly this format with no extra text:\n\n'
                f'WHY: [One sentence, max 15 words, explaining what specific expertise makes this client relevant to this headline]\n'
                f'PITCH: [2 sentences, under 75 words total, written to a journalist on deadline. Lead with the angle. End with a natural, varied close. No "unique perspective", no "subject matter expertise", no "as a leader in", do not start with the client name]'
            )
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=150,
                temperature=0.4
            )
            return response.choices[0].message.content.strip(), f'Groq / {GROQ_MODEL}'
        except Exception as e:
            print(f'  Groq error: {e} - using template')

    # Template fallback
    topic_display = top.get('topic', 'this topic').replace('_', ' ')
    if is_calendar:
        pitch = (f'{client_name} is focused on {topic_display} this month and can speak '
                 f'to this from a community health perspective, including patient outcomes '
                 f'and access to care across Arizona. Available for comment.')
    else:
        pitch = (f'{client_name} has direct expertise on {topic_display} and can speak to '
                 f'the specific legal and safety dimensions of this story — not just '
                 f'background context but actionable perspective a journalist can use. '
                 f'Available for comment.')
    return pitch, 'template'

print(f'Pitch generator ready (model: {GROQ_MODEL})')

Pitch generator ready (model: llama-3.3-70b-versatile)


In [18]:
# Cell 11 - Export digest to JSON for web UI
#
# DESIGN DECISION - headline deduplication combines two signals:
#   1. Same topic tag - ensures we only compare headlines in the same domain
#   2. Bi-encoder cosine similarity >= 0.55 - ensures semantic overlap
#   Both must be true to collapse two matches into the same story group.
#   This mirrors the hybrid search architecture already built - combining a
#   structured signal (topic label) with a semantic signal (bi-encoder).
#   A single threshold on similarity alone is fragile: a different drowning
#   incident in a different city could score 0.80+ just from shared nouns.
#   Requiring topic agreement as a gate prevents false collapses.

import json
import numpy as np

def parse_pitch(raw):
    why, pitch = '', raw
    for line in raw.split('\n'):
        if line.startswith('WHY:'):
            why = line.replace('WHY:', '').strip()
        elif line.startswith('PITCH:'):
            pitch = line.replace('PITCH:', '').strip()
    return why, pitch

DEDUP_THRESHOLD = 0.55

def same_story(m1, m2, emb1, emb2):
    same_topic = (m1['top_results'][0].get('topic', '') ==
                  m2['top_results'][0].get('topic', ''))
    sim = float(emb1 @ emb2)
    return same_topic and sim >= DEDUP_THRESHOLD

def group_by_story(matches):
    if not matches:
        return []

    sorted_matches = sorted(matches, key=lambda x: x['score'], reverse=True)
    headlines      = [m['headline'] for m in sorted_matches]
    embeddings     = model.encode(headlines, normalize_embeddings=True)

    groups   = []
    assigned = set()

    for i, m in enumerate(sorted_matches):
        if i in assigned:
            continue
        group_similar = []
        assigned.add(i)

        for j in range(i + 1, len(sorted_matches)):
            if j in assigned:
                continue
            if same_story(sorted_matches[i], sorted_matches[j],
                          embeddings[i], embeddings[j]):
                group_similar.append({
                    'headline': sorted_matches[j]['headline'],
                    'url':      sorted_matches[j].get('url', ''),
                    'score':    sorted_matches[j]['score'],
                })
                assigned.add(j)

        groups.append((m, group_similar))

    return groups

export = {}
for client_id, matches in all_matches.items():
    strong_groups = group_by_story(matches['strong'])
    adj_groups    = group_by_story(matches['adjacent'])

    export[client_id] = {
        'name': CLIENTS[client_id]['name'],
        'strong': [{
            'headline':    g['headline'],
            'url':         g.get('url', ''),
            'score':       g['score'],
            'stale':       g['stale'],
            'topic':       g['top_results'][0].get('topic', '').replace('_', ' '),
            'is_calendar': g['top_results'][0].get('is_calendar', False),
            **dict(zip(['why','pitch'], parse_pitch(generate_pitch(g['headline'], client_id, g['top_results'])[0]))),
            'similar':     sim,
        } for g, sim in strong_groups[:5]],
        'adjacent': [{
            'headline': g['headline'],
            'url':      g.get('url', ''),
            'score':    g['score'],
            'stale':    g['stale'],
            'topic':    g['top_results'][0].get('topic', '').replace('_', ' '),
            'similar':  sim,
        } for g, sim in adj_groups[:5]],
    }

with open('digest.json', 'w') as f:
    json.dump(export, f, indent=2)

from google.colab import files
files.download('digest.json')
print('Downloaded digest.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded digest.json


In [19]:
# Cell 12 - show top 3 matches
for client_id, matches in all_matches.items():
    strong = matches['strong']
    adj = matches['adjacent']
    all_m = strong + adj
    all_m.sort(key=lambda x: x['score'], reverse=True)
    print(f"\n{CLIENTS[client_id]['name']} top 3 scores:")
    for m in all_m[:3]:
        print(f"  {m['score']:.3f} | {m['headline'][:60]}")


Client 1 top 3 scores:
  0.711 | Dillon Brooks DUI arrest spotlights Arizona’s marijuana impa
  0.637 | Child Miraculously Woke Up in Morgue Hours After Reportedly 
  0.621 | Arrest of Phoenix Suns player draws attention to Arizona DUI

Client 2 top 3 scores:
  0.718 | Who’s Who in Arizona Healthcare for 2026
  0.648 | Spreading awareness for National Kidney Month
  0.647 | Agenda Posted: AZPHA Conference - From Crisis to Care: Impro

Client 3 top 3 scores:
  0.719 | So. NV Nonprofits Receive $75,000 Cox Charities Grants
  0.638 | Arizona engineer helping families in Iran during internet bl
  0.612 | Cox Communications awards grants to Southeast Kansas schools

Client 4 top 3 scores:
  0.611 | AZ Big 100: 50 Arizona business leaders to watch in 2026
  0.578 | AZ Big 100: 50 commercial real estate companies to watch in 
  0.532 | Arizona Considers Legislation to Deter DEI Programs and Poli


In [20]:
# Cell 13 - Weekly digest

today_str = datetime.now().strftime('%B %d, %Y')
lines     = [f'# PR Opportunity Digest - {today_str}\n']

for client_id, matches in all_matches.items():
    name   = CLIENTS[client_id]['name']
    strong = matches['strong']
    adj    = matches['adjacent']
    lines.append(f'---\n## {name}\n')
    if not strong and not adj:
        lines.append('_No pitch opportunities identified this week._\n')
        continue
    if strong:
        lines.append(f'### Strong Matches ({len(strong)})')
        for m in sorted(strong, key=lambda x: x['score'], reverse=True)[:5]:
            stale_flag = ' warning - older story, confirm still relevant' if m['stale'] else ''
            top        = m['top_results'][0]
            source_tag = 'Calendar pitch target' if top.get('is_calendar') else f"Matched topic: {top.get('topic','').replace('_',' ')}"
            pitch, method = generate_pitch(m['headline'], client_id, m['top_results'])
            time.sleep(2)  # stay under Groq free tier rate limit (30 req/min)
            lines.append(f"\n**{m['headline']}**{stale_flag}")
            url = m.get('url', '')
            url_str = f" | [link]({url})" if url else ""
            lines.append(f"Score: {m['score']:.3f} | {source_tag} | pitch via {method}{url_str}")
            lines.append(f"\n_Draft pitch:_\n> {pitch}\n")
    if adj:
        lines.append(f'### Adjacent Matches ({len(adj)})')
        for m in sorted(adj, key=lambda x: x['score'], reverse=True)[:5]:
            stale_flag = ' (stale)' if m['stale'] else ''
            top        = m['top_results'][0]
            url = m.get('url', '')
            url_str = f" | [link]({url})" if url else ""
            lines.append(f"- **{m['headline']}** (score: {m['score']:.3f} | topic: {top.get('topic','').replace('_',' ')}){stale_flag}{url_str}")
        lines.append('')

display(Markdown('\n'.join(lines)))

# PR Opportunity Digest - March 11, 2026

---
## Client 1

### Strong Matches (4)

**Dillon Brooks DUI arrest spotlights Arizona’s marijuana impairment laws**
Score: 0.711 | Matched topic: marijuana driving | pitch via template | [link](https://news.google.com/rss/articles/CBMiqwFBVV95cUxOMC0tX2FyLVh1d09IcXd3N2xRMWxUNzhkLVdGNU5OZGdWQy1PQ2dlbXZOdE1tQndRWnNJWjRYb1lrS1Ewelc0N0ZTZU9vQktEdmJ2Rk1HWGtROTdPRXZxcXFCQVNKa3BxalRPRW9YN25NRVZTdFBuUVUwRURDeFFNM2ZuS1M5N1FYSU1YRzN4eE42SDNpSHl5VmlXV3dDZmowdW03SVBhSll5ZTjSAb8BQVVfeXFMUHdWWmhOOTEtcms2TlFKeGtieUlEc21xTmwxTEgxMXlLNWluc2lnQ0pVbnh4R2duZ3FmTnJSMXA1eFVmbjdqZjkxdXdfQThMVElqd2dGYUs1bld2ZWJyT0o5UmNsWkRPUFBteTB0bTExRDFOT3I5eEltbWRBcFcwTE5pVXg4X2pNcHBKWldPVmluMWZrN1p1SV9qcEVEcHo5dEx2bmJMVnpzLTRscF9Dd1Y4ZVJVTHhhT0pISVZSV0k?oc=5)

_Draft pitch:_
> Client 1 has direct expertise on marijuana driving and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.


**Child Miraculously Woke Up in Morgue Hours After Reportedly Drowning in Backyard Pool**
Score: 0.637 | Matched topic: pool drowning safety | pitch via template | [link](https://news.google.com/rss/articles/CBMiowFBVV95cUxNaVUwRlR3OTN0YkpucTctSUY2bE54VmFDS0ZCODZaUkRiSWxzR2hCN2RYSEVfUTBhSG51c0JvaXR1aEhhaXROUmZxSVptdVJmUGpBUHhVUE1Wcjh5RWhLSk05Qm0wRlk5LWVPQTA3NVprX3VJazdveEQ2SHV2MTNxYVR4OFRIZHROalp4WklteTlJdkpnOGdGaW5VUTdJWFBYTW4w?oc=5)

_Draft pitch:_
> Client 1 has direct expertise on pool drowning safety and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.


**Arrest of Phoenix Suns player draws attention to Arizona DUI laws**
Score: 0.621 | Matched topic: marijuana driving | pitch via template | [link](https://news.google.com/rss/articles/CBMi2AFBVV95cUxNME9yVXYydjQtTm9Ta29ELXEwMjVxeG9xamp5aXB3SFVjSEN0MTYzNEd5YWxheG5GQXowZlRiOUhJYV9mSzN0OEdxZ2pySkdkMkVKR1N1Qkw4eUNGXzhIanNISTFKR001NU5VLXJQSXpnQjhVSkt4b2NzN3NTa254QVJTYVlnZU9Id2d6b3U5U3pIWDlCS1p4Mk1KTkFfN1Q2UWRsWnlMS0podEFnd2RoNzh4ZmFYY3hsUDFXRjZmeS1SWk9rSXFjamluekNRMGFGRzBDU3lzR1E?oc=5)

_Draft pitch:_
> Client 1 has direct expertise on marijuana driving and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.


**Dillon Brooks' arrest highlights lesser-known marijuana DUI law**
Score: 0.584 | Matched topic: marijuana driving | pitch via template | [link](https://news.google.com/rss/articles/CBMizwFBVV95cUxONUlsNlF6S2JYclJzOUxmYWVaYW1IVWZpUHEzS0FBb1JRYkhRRFRwbEpmb3Bfd0NDUHRkVWx3NDRYaU9xcUp0SVlOQ0xMb0lKS3RMNHp0cjZoTzFLc0JpLXpPdHVsLTZ3a1o5MEtlQkNKZGlsOFJ1dC00WHNCNlU1b0hkYjNnc1p0WGNITF82WVBLLXJtSlMtRnF2LXNUMXNZVkdGX2N3Mk5GY3RmLTJVbzg0RXNid0Frc2pkVlNQYUlTWUV6Nk16WkFOZ1RzU28?oc=5)

_Draft pitch:_
> Client 1 has direct expertise on marijuana driving and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.

### Adjacent Matches (9)
- **Amazing: AZ child cheats death after pool | 830 WCCO** (score: 0.550 | topic: pool drowning safety) (stale) | [link](https://news.google.com/rss/articles/CBMibEFVX3lxTFB5dloxTzhESGc1M2RadFQwR2stRHJBU0dfOXhVT0xXVzdTc296SzFtMHUtNktmNllYV0JkMG51b19YQXNjLVdXQUJpaXRDTVo1NTlxTFJTQUNyakhKdnp5R3VEUnFpSWZMM1o0ag?oc=5)
- **Arizona mom killed while protecting child from dogs** (score: 0.532 | topic: dog bite) (stale) | [link](https://news.google.com/rss/articles/CBMi2gFBVV95cUxOamViSzFkaGJEU2tUY1hHT3hiODFBREdDaGVraEVpZDhDaWRqRnpZMGZvV3FGd000MWpmcUN1RklHdXZ3QjE1ZU92QkVMcm5UR3l6ekl0S0poZmVTb3E4Y3dpX2lBUEd3TklUajFudEtOQk5oU2lrVDZHLWYzT2RaT0N0NDlRaHIyNzAybXdRVEcxeVNrdkNWWnlwX1J2QjNIWUU2TWFsYXBob1hPNUF4Y2puaWtLZWxJRTZXeS1lUm5KeWRyZldya3h4TmhQandDVnRyVnpNdWFnQQ?oc=5)
- **Arizona Mom Dies in Dog Attack Shielding Her 5 Year-Old Son** (score: 0.530 | topic: dog bite) | [link](https://news.google.com/rss/articles/CBMipgFBVV95cUxNTlVsaTlTNkRMTERYbW1WY0Nhb2tvTnVub2FTd1ZuWXhTQmExTjBibmhUUTdoVWdJMFUxLUdhN3RRTzFwX0wxZzJaajNwWGJfYzdYU1BxRVhZeWFZaFRDWXotVmg5WWdRRkdadmxMVWh2YzRnV3czSkpvdUozSkRfTHRLQnRENTZONkk5VEJXRU1yQlJLb3l6NVJOY1pSNlVCNzNxWHdn?oc=5)
- **Arizona Mother Killed While Shielding Son During Dog Attack in Southern California** (score: 0.516 | topic: dog bite) (stale) | [link](https://news.google.com/rss/articles/CBMitgFBVV95cUxPUWxMT29WUHZ1Y1U4OXFYa05HUXlzLVkzdzVlSzBJWHJMWXBwV1VKTHlsQjA2Yk1jemNpZXVZMlhiOGlQZFpUQ0lJT0dQaGZ6eEloSUhmLTd2OVJPYlBNM2pvTGh1VUdibFQ4UngyS2N4VVJVUFc3UGtNQVY3dGhueHA5VDRKSWQ1MEowOGdyaFY5M2tpVTk4d3NuSG05QTRRRWR6elJ4RlR5dUM2MDFrR2RoWWNyUQ?oc=5)
- **Arizona mom dies after shielding her son from a dog attack** (score: 0.510 | topic: dog bite) | [link](https://news.google.com/rss/articles/CBMivwFBVV95cUxNazFqeWFJTlBXaGE1c1V5LTk5U3E1UmVuMHJLVkVaNDcwcWhUMFhyREtvVHNGa3B3NnRGN0dPTXVfb0t1N0g5eHZsU0tzV1ZuS245QlJGZzRhUmNxMjhTS25jZU5XY0xmVWczUkJucFM1VWxyenN2dW9YYVFqMWtFWjlydjdadS1xWWlPTkZ5V3pGaUFzVHdnbVc2SHR5U1NjZUltOFZZVWNfOTVpTVZKR3RibmZGeC1raHIzLXBhUQ?oc=5)

---
## Client 2

### Strong Matches (7)

**Who’s Who in Arizona Healthcare for 2026** warning - older story, confirm still relevant
Score: 0.718 | Matched topic: cancer awareness | pitch via template | [link](https://news.google.com/rss/articles/CBMijAFBVV95cUxOV1NjZ3JvblJhSjhiSzZxRE5GT0pIZ1lzWFU1VTUyUnVrQmd6ZHpfRlVlSXdkdjlndWMtOTIyVmRENHdqWkplRDBFVkd3NFZUYmFPT0pQZmFqQUU2d19xM3ZGT0phQWNzSTNpc0l1MWpIcGlVX0NSdmlNLWluYTg3cVZNWWYzUTZaNnVMVg?oc=5)

_Draft pitch:_
> Client 2 has direct expertise on cancer awareness and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.


**Spreading awareness for National Kidney Month**
Score: 0.648 | Calendar pitch target | pitch via template | [link](https://news.google.com/rss/articles/CBMiwAFBVV95cUxQOE12aGRLTTdyT0czdnYyWUJxamoxaHNFQnhQRk5La0NMTmpWcGFKdmpOSktYc051Y004Z2tRT0NyalNHRnEzVTh0MWpnNHg4V2hiZndpckhUbHlEajU0dUpFckxTZm5nX212SGlxTUEzZ184dnVsc0l4NG1ocU1OTnpPRzc5ZXBRR3ZDVXJQUVdJdDlWM2JlQWlQZy1VRWhHaV9wdlpSNWMydW9ISEFxbVpLWXhxOC1PYTN6RDJydi0?oc=5)

_Draft pitch:_
> Client 2 is focused on chronic disease this month and can speak to this from a community health perspective, including patient outcomes and access to care across Arizona. Available for comment.


**Agenda Posted: AZPHA Conference - From Crisis to Care: Improving Outcomes in Arizona’s Behavioral Health System**
Score: 0.647 | Calendar pitch target | pitch via template | [link](https://news.google.com/rss/articles/CBMi0gFBVV95cUxOUlRzdHBDSmgzNmFETjBEWGZVbDhxbTJFajNFVDA4cnNLeHo2dFJDRVloT3ZvSE84WEk4Tks5M2xkMzNwU3d2WU5oNFcwMjR6R2ZobkJoX08yVGVHUm9QcGVCc1JDWjJITzNBWVZEVUg2RlNfSjNWaVBsbHdhNGRiNnBQNFBmSFNraHdsaTBhSHkwX3BKTThWeFJBd3lpZVdHYVVaTi1jeVlvQmJ6ZUpSVFV1UHNOWjV2Q0xCR2htWHFVVEI2QXMyaUI1MjJvU0wyR1E?oc=5)

_Draft pitch:_
> Client 2 is focused on mental health this month and can speak to this from a community health perspective, including patient outcomes and access to care across Arizona. Available for comment.


**Colorectal Cancer Awareness Month highlights importance of early detection and screening**
Score: 0.613 | Calendar pitch target | pitch via template | [link](https://news.google.com/rss/articles/CBMixwFBVV95cUxOckFZRDh0VkxxMG1zbUhmbjVpcGNfOWQxUWdSQktlS1FRYVpCekNHT2NLRkg5THpSSkxkTFFmUEZZYTFKRlJObHRCMGE3WW1jdzVFTXdrOWNwYk9wNS1MbFBvUE9wekNVODAzUHhDMy1GY0c4Nk5kYk5janpQSVE2cDMwNFU1cE5rSHowbjF0V0ZYVWFNV3ZkWjhHOVphbU5aTlJ3WnJBX2hvZ19abUw1LUZHYmVnVmNodDdLdUtHdFU3OWRQODFZ?oc=5)

_Draft pitch:_
> Client 2 is focused on cancer awareness this month and can speak to this from a community health perspective, including patient outcomes and access to care across Arizona. Available for comment.


**Talk With the Doc: March is Colorectal Cancer Awareness Month**
Score: 0.604 | Calendar pitch target | pitch via template | [link](https://news.google.com/rss/articles/CBMiuwFBVV95cUxORTU2OFN5OWZPWVl2SHUxSDF3Y3ZvbVF5b3ZBaTRQLXh6d2w0dDhxX1lvcjhnV3A3WXczLVQ1TG1TSk9GM1Y3RlloNEh2NVctanplV1ByVU5nT011NjF2MDJSaVpleWt1UlJEczFHd2E5NFFndFhITXlCSlpueTdQTG5jTTVxNkJtRERGdjlXR0haVjlYeXJSejhuek14bHBnLW5wS2ZOZC1mLTFTUElHVHRQRVlvR0VFdkFB?oc=5)

_Draft pitch:_
> Client 2 is focused on cancer awareness this month and can speak to this from a community health perspective, including patient outcomes and access to care across Arizona. Available for comment.

### Adjacent Matches (13)
- **Arizona lawmakers should finish the job on expanding pharmacy care** (score: 0.545 | topic: mental health) | [link](https://news.google.com/rss/articles/CBMirwFBVV95cUxPaDc5RHR2cWlUeGV6NnNQSlJYU2xCekZwOTR4YWFWZGo2dE1PNHBuVnRvLUp3LWtRN1FEdlNuRzVGeHNlNnlhUGNNb18wdzRHYkFsaXJmeTJNMDBpSWl6NmxDMGpld0t2ZS1obmI1MzJieEh6aWJPTV9jS1pha2tyWGx4UGJqcnYxWDY0em9CcWJPMlh3NnIzM19jc05BaDFwNVgzc1ZCSHUwbHh4Z3I0?oc=5)
- **March is National Multiple Sclerosis Awareness Month** (score: 0.523 | topic: chronic disease) | [link](https://news.google.com/rss/articles/CBMijwFBVV95cUxPWG5oWUJkTWJrUTFVR2k3SGlWNUNadnVnWTJ0UV90SVpEN3RWTUdhLWtfaXNMMVVvbm14dFdUSkdORjltMXdldWdsNmFZMHM0UHZJVlFyRXJnRDJCendXaVhScEhfUUNOeVBJZVo3ZkNaRHhiLS1LWXNhS2JSSXluX3BCUVdHN2N1b1dzd0EtSdIBlAFBVV95cUxOaTdmMHN4ZklERFBiUGdGbVVLQnlHdXBtdmhqcDhnNC1ZekZhdUlQeHpsZ3dNdHdCUHZoejBBeXVXWGFUSjZBV2dpSjM1dGZ1NExmRC1HOVU0ejBUelpjSGR1Y0tJdF9UNnhpZmJyUm96UmQxOGZDdmpLQXB2YkY5V3hNTzBOY0ZDYmxEUGtpOFVFckxN?oc=5)
- **Hidden epidemic: Brain Injury Awareness Month shines spotlight on Indiana survivors** (score: 0.513 | topic: cardiovascular health) | [link](https://news.google.com/rss/articles/CBMib0FVX3lxTE1tZzlFclNrWGxkcGlORTU0dDJJY0w2OHhpQ0Q2ekJ5TG5PbUpkeS1Sa1BVajFQYk1iSEdwT0RKVzU5R3NfNF9zcjRKelBJMlctZXFsNGtpck16SVNHc2dmbWhTU2NtZTlUeGdRZFZaQQ?oc=5)
- **Arizona Senate passes bill targeting doctor pay for vaccine rates amid measles surge** (score: 0.512 | topic: infectious disease) | [link](https://news.google.com/rss/articles/CBMiswFBVV95cUxPUjJiWVNneERWMEQzQlBTdlRIVmxoMmZSOVRFc3JNQmpRREcyNTZMYUxsamNBcXZLdVJUUHNUMExkTFlMSHBSMnJrd1BGcGVFTkhIQ01xdXVPNjJZS2NNVzdITW1Da0dIUnRMc2FCbk9SSmpOeHZzODNxOWNMSzVVU2ZOLURJb3VPcnZoWUI1QzJFbjFRdWxXRzdFcFBmVnRTZ3d0SVVlQ1BKcldRb0dHd2xpdw?oc=5)
- **Arizona senior communities are using AI to prevent falls; here’s how it works** (score: 0.504 | topic: cardiovascular health) (stale) | [link](https://news.google.com/rss/articles/CBMirgFBVV95cUxNWmczdUxCZndqVlpudUNaWV9wT2FrVXJOV3ZwanBVN0c5TVo3ZWxxTHMxd1Mwd3pwS2JGMHpQM2Y4WmZNQWhobE1ubmswNDI5YV9VSkFYRDlMZTJJZlo0ZlVwcjJiYzJQcXVsR2ZNX0sxYUlPTHdTZXdaWkxpYzQtc3dQbmlOSWlqaFhOS3FKVWRFNUVWc25wVFR5SWFQd2dENVc5RWFjcmVoOEM1RWfSAcIBQVVfeXFMTW9YNmlTcW1KMFdaVG1xcldTb0RIa0x0bEduRkpOVUp0VEM4Ui1xai1aMy1jd2ZDUlNnU2FDTXVBSm5sUVZuSC1zS2RqSXhHUExNSk5DRmp2WndfRXh2Q0J4eXl5WkprbTFHVUg4dF9naG5ib3dXRF9rZ3NJa25EQWc2MG41bDhSUGZjaUJGU3dyLTFWa05RT1F3dHBRdk51UHZhTmstRHl3TW5PM0daMXpPTjJNMmxpdTl0LUhLdHVmc0E?oc=5)

---
## Client 3

### Strong Matches (4)

**So. NV Nonprofits Receive $75,000 Cox Charities Grants**
Score: 0.719 | Matched topic: community giving | pitch via template | [link](https://news.google.com/rss/articles/CBMikAFBVV95cUxNSk9xV1Q4Wnkwd3FCV3pOUzA4cjI2V3lOWmdTcVV3eDIya2JkMGk3RFpHYVB2Ty03bklBWXlCbkpCdEpsWWdISk9JeVhOeXAzWmJqZVlJeER0YV9JRDA5WWEybDRlNHhIY1FOa1g0XzlURGVSc2JVOElxU0JZbEpnNjI2RTRicnhORi1hZTZOaVA?oc=5)

_Draft pitch:_
> Client 3 has direct expertise on community giving and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.


**Arizona engineer helping families in Iran during internet blackout** warning - older story, confirm still relevant
Score: 0.638 | Matched topic: connectivity | pitch via template | [link](https://news.google.com/rss/articles/CBMiqgFBVV95cUxQQlFfY3dXSDAzRDdGY0hoY0t1a0NfMk5jRzZPWE13NF9rbHNBTndiYUw0UldZeE1taWNRV3o0MEZVdXpZM0lRRHJSb1VhZVFYVXJQcEsteDhmNnZvYWNlTnRyOW9La2ZIX3hjdHdXbDBubWhkb2g4SG9vS0xkZ2ZxVnNwcTA0LXl6a1haS29ZdG9JWHJ2M1V4SzFGbEVQTFdLVGxwUVpvSmxnQQ?oc=5)

_Draft pitch:_
> Client 3 has direct expertise on connectivity and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.


**Cox Communications awards grants to Southeast Kansas schools** warning - older story, confirm still relevant
Score: 0.612 | Matched topic: sustainability | pitch via template | [link](https://news.google.com/rss/articles/CBMi6AFBVV95cUxPak9Lb0VkWU1iYm5QTTRJMVhPSkhQUFdGWFhDTFM2dzFncVlvN3UxZ0ZHV2ZJeHJucm9HWWRrLUpYaUZYMlRlTFkwc3N0bnFCZGpieGNVNlRiMFZtTmpINE1Xc0FaYXZ5Z3NMWjFTSTFpT1pFSjF5N3RmZV91emlGNzA4WWlvMzRNZ3JNVzV4bDRqcEdjU1ZtV2dHZkZEd1BWODliM2U5VkZHNTVuU0lXUVh1YUt3OUh1UDNVUkVhX2ZLY1ZzckQtVkQ2WE9NRDhOUlZ3WC04R2psQzBmZDlKX1FVUVZFZmRv?oc=5)

_Draft pitch:_
> Client 3 has direct expertise on sustainability and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.


**Arizona software engineer helps Iranians bypass internet blackout with free VPNs** warning - older story, confirm still relevant
Score: 0.561 | Matched topic: digital equity | pitch via template | [link](https://news.google.com/rss/articles/CBMiuAFBVV95cUxPNTRxa3JSY1BGay1GVC1EOWJITGo1c3pQbWRSVFFyM0pVd1NObTdqcERPSFR6QTRJQzdfQk5jMWp4OEZoTjZlamtzT0JocHpWM1VJN0poMC1SeXI3X0NBNGxxcUFsUUpLMzlTWm5JeTNNajVZT0lEMS02V0RBaXA5c19jRzZPckdRWjdLWk8tOUd4dEFCamZhOUhzLXN5RmNBZjIyaVVGWVEtc0FVNDFTTDB6REFVdkpQ0gHMAUFVX3lxTFBUeHB3dGM4ak9pTVluaU53M0dkRFg2NWFxbTJucWZmWnJkYmNRQ3BHVzlSYzRmd3JfeEtSMUVpV1V2N1E2cFcxMFdwdTRKb3FqSnlXeXViOHRLdDhSZ09kVEJwWWV6dEtsTkp4RHRRV2lycE01UGlpT3Y2N0dYOHhCbmJneE15WkloLWYwbXdHNlBhX0FudWdsSktQSXR5eGU5YzdOd0J5M2d4VkVIODYwUWdGWHFQbHlHZy15Q1hNSVBtcklvTnQzdndJag?oc=5)

_Draft pitch:_
> Client 3 has direct expertise on digital equity and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.

### Adjacent Matches (2)
- **Charter Communications slips as Cox deal timing and leverage concerns resurface** (score: 0.503 | topic: broadband connectivity) | [link](https://news.google.com/rss/articles/CBMisgFBVV95cUxQaUpXaFNGTEVqeDRaa0trdkprdHZ5LWZXRU9OQ2xhRzUzNktJNGlQNTVLYTQ1OXdmYmk3ZWRKbU5ZNWZqd3lpMktGMDFIckl0d1pzYThud1RST2ZvMjlrSTZLd1BFUVJFNHI5YkdRSFVDYWxRQmxkUkpTZFMtR3pCbmNfdjVua1UyM2xoRTd1c3hPOG0wdlo0ZlA1S1pqY2JxVnlJc3dXUGtuWGJYWURESzZB?oc=5)
- **Google Fiber coming to Apache Junction** (score: 0.414 | topic: broadband connectivity) (stale) | [link](https://news.google.com/rss/articles/CBMidkFVX3lxTFBIeHNuNWd4SnNjM2xDeU96RnV3alpDWjVsSXNveUhSVERyOGNUbU1YMTNQUUhoRUZzWDFJam4wMGlkaW85UDhLV0pfaUFMQkNMSmU1QWFnMzVXRW5oUk81bzE0QTZTczNicTM2Q09UODNaWWVLSmc?oc=5)

---
## Client 4

### Strong Matches (2)

**AZ Big 100: 50 Arizona business leaders to watch in 2026** warning - older story, confirm still relevant
Score: 0.611 | Matched topic: regulatory compliance | pitch via template | [link](https://news.google.com/rss/articles/CBMikgFBVV95cUxNd2Z1WWhSQUxtVkRiWVZWVFI3UUxMeWowODJ6MGxjbTB6ZjBmNU9ZTDVudWpJNHdHOGJoWGtoQUFSSkx0U2RfWFVWYnAxX2JWbXFjNUpPaUZIdnAwRzZ5a1pPeFkxS0ZWYmNDeDdiSzZ6MWtXSDd5YzBmanN5ZmxyanBwNmpVU1I2N3hURTRyNXhnQQ?oc=5)

_Draft pitch:_
> Client 4 has direct expertise on regulatory compliance and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.


**AZ Big 100: 50 commercial real estate companies to watch in 2026**
Score: 0.578 | Matched topic: regulatory compliance | pitch via template | [link](https://news.google.com/rss/articles/CBMioAFBVV95cUxNVnF4U19RZGt3aEpPRnlndzBKbmVqNTg1Z2pRdUh4YkFRRkVKV0FWYk91dWRHcTdrd3Bma3Y4NVZzclZnUmtfVnZXcWZvY1VsTWQ3dU1OR3c0ODBFRnF1RHNwZkhPVHdfRG9nRzVoMFhKTzVMMWgzNVE4R29JRnFYTG5NOFNZaXBGZGd0YjhIWEg0U1dNSENfd3ViSmloUERa?oc=5)

_Draft pitch:_
> Client 4 has direct expertise on regulatory compliance and can speak to the specific legal and safety dimensions of this story — not just background context but actionable perspective a journalist can use. Available for comment.

### Adjacent Matches (7)
- **Arizona Considers Legislation to Deter DEI Programs and Policies** (score: 0.532 | topic: regulatory compliance) (stale) | [link](https://news.google.com/rss/articles/CBMitwFBVV95cUxORDVTLTh3QWdhR0RDUmdmNjFHVy1KTE4xeVZQOU1keDVUTXhwZ2VuTHFYbnhpVmJ5dlpDZE1HejNGTGtkWWNibmtsUEM1a2pqSDhNYlgzaHZMVW5EVXJkY0ktNlp4aG5qcng0YlBqTGdWRjNEODFlZ2xYNXhweEdhaF93cmRZSXh0d1IwS0NOTXNnS0tnZUo1T2xzdGZYcEtqeUpqZ0dTR1JDa3EySXBhZUJHSnhaazQ?oc=5)
- **The Top 100 Lawyers in Arizona for 2026** (score: 0.517 | topic: regulatory compliance) | [link](https://news.google.com/rss/articles/CBMifEFVX3lxTFBMeVZiY1QxaXcxZE5zbXNHalduamF1eHpNNi1TU1F1VzBrZnBjN2NHWDJBeXA2Q3FSa3hTSVJYSEs5QXh3d2lINktRU2ZQaHR0Tk5hbGFxZDVrZDVWS3NRRWx1T3pCZ08ySGl4QkVBdVprdUtDdVFwbm9HcUo?oc=5)
- **Arizona among 20+ states filing lawsuit over new global tariffs** (score: 0.508 | topic: regulatory compliance) (stale) | [link](https://news.google.com/rss/articles/CBMimwFBVV95cUxPWFN6bnNfaC1kLUl0SFpoV2JiODFFYkk4UG9LcEkyemh2UklveUYyYzJXb1kwY0xNZGlKb29WeWhfdU16aFN3dTZjSEtXTURvX1UyV0RRbDBFSFNxU2E3UzM4SzZZWTVUdEZ2cG1LMVdfZGNfRllHQ3FMTW9zSHJSZ0ZXQ1NwOWs5YjJUS2JnS0MwdXJhOUk2QmFWWdIBrwFBVV95cUxPel9SSzI4Ykp6U2pzaEp5VHRsYWFXVV9TZVRmNWYyeFI5TWhUVmswZ3hxVDhZdk8tVHdfVjZVeXNBN3ZMVVgwMWNob080cGdBVTUwWklYM1JUNVZhdUNPNjhfZzFxLTdSbXpIMElXRm1pRG94cXVJOEpXYjV2cFhQZ1N4NU1faE8zYnVEd3hOSk1HMktkM3hTQW0zb3FiMmg1YmN0clcwVzNGM0kwMHVv?oc=5)
- **Trump's tariff plan faces lawsuit by Arizona, other states** (score: 0.495 | topic: regulatory compliance) (stale) | [link](https://news.google.com/rss/articles/CBMidEFVX3lxTE05SWhaaE5JOWwzWVhlbXNKWWVYbUpZZzV6RllHblN0WlRxd0JVVVZnYXpQb1J6RTZWWDNmUUpTbVNua3U4TUJ6UTRYanQ2cURya1FWNWRIZEVhSnVyTzNpN1BWMENUaDZ4UXFvejZTamZCU0xH?oc=5)
- **Phoenix developer Vestar eyes two state land parcels for mixed-use projects near TSMC** (score: 0.477 | topic: commercial real estate) | [link](https://news.google.com/rss/articles/CBMipgFBVV95cUxOa09NQXN5VjBQZVNEbF9qY0xTR3lKNVdvUmRQNTZlazFoZk9XSTVwRkV6RFlDeWVwUm5fR0kwUHNBU0VsWXgtSWd6NVVobmN2NjktSmd4RjREaERHNTdmemlMUEZ5bkRKU2ZDcGluNHM2SHdaNXQzRUZqSDVvdVFHWkNNUG1EcXk1SVJKLTA0N21vMXlvaFdfUXhlelZuczFzVUpXRXln?oc=5)


In [21]:
# Cell 13b - Export digest to Word doc for human review

from docx import Document as DocxDocument
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

def add_hyperlink(paragraph, url, text):
    part = paragraph.part
    r_id = part.relate_to(url, 'http://schemas.openxmlformats.org/officeDocument/2006/relationships/hyperlink', is_external=True)
    hyperlink = OxmlElement('w:hyperlink')
    hyperlink.set(qn('r:id'), r_id)
    run = OxmlElement('w:r')
    rPr = OxmlElement('w:rPr')
    rStyle = OxmlElement('w:rStyle')
    rStyle.set(qn('w:val'), 'Hyperlink')
    rPr.append(rStyle)
    run.append(rPr)
    t = OxmlElement('w:t')
    t.text = text
    run.append(t)
    hyperlink.append(run)
    paragraph._p.append(hyperlink)

def export_digest_docx():
    doc = DocxDocument()

    title = doc.add_heading('PR Opportunity Digest', 0)
    title.alignment = 1
    doc.add_paragraph(f'Generated: {datetime.now().strftime("%B %d, %Y")}').alignment = 1

    for client_id, matches in all_matches.items():
        name   = CLIENTS[client_id]['name']
        strong = sorted(matches['strong'], key=lambda x: x['score'], reverse=True)
        adj    = sorted(matches['adjacent'], key=lambda x: x['score'], reverse=True)[:3]

        doc.add_heading(name, level=1)

        if not strong and not adj:
            doc.add_paragraph('No pitch opportunities identified this week.')
            continue

        if strong:
            doc.add_heading('Strong Matches', level=2)
            for m in strong[:3]:
                stale = ' ⚠ older story' if m['stale'] else ''
                top   = m['top_results'][0]
                source_tag = 'Calendar pitch target' if top.get('is_calendar') else f"Topic: {top.get('topic','').replace('_',' ')}"

                p = doc.add_paragraph()
                p.add_run(m['headline'] + stale).bold = True

                score_p = doc.add_paragraph(f"Score: {m['score']:.3f} | {source_tag} | ")
                if m.get('url'):
                    add_hyperlink(score_p, m['url'], 'link')

                pitch, method = generate_pitch(m['headline'], client_id, m['top_results'])
                p2 = doc.add_paragraph()
                p2.add_run('Draft pitch: ').bold = True
                p2.add_run(pitch)

                doc.add_paragraph('Relevant?  [ ] Yes    [ ] No    Notes: ___________________')
                doc.add_paragraph('')

        if adj:
            doc.add_heading('Adjacent Matches', level=2)
            for m in adj:
                stale = ' (stale)' if m['stale'] else ''
                top   = m['top_results'][0]
                p = doc.add_paragraph(style='List Bullet')
                p.add_run(f"{m['headline']}{stale}").bold = True
                p.add_run(f" — Score: {m['score']:.3f} | Topic: {top.get('topic','').replace('_',' ')} | ")
                if m.get('url'):
                    add_hyperlink(p, m['url'], 'link')
                doc.add_paragraph('Relevant?  [ ] Yes    [ ] No    Notes: ___________________')

        doc.add_paragraph('')

    fname = f'PR_Digest_{datetime.now().strftime("%Y%m%d")}.docx'
    doc.save(fname)
    from google.colab import files
    files.download(fname)
    print(f'Downloaded: {fname}')

export_digest_docx()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: PR_Digest_20260311.docx


In [22]:
# Cell 14 - Human feedback collection
# Captures expert ratings - the human preference signal from the RLHF/DPO pipeline.
# Future work: use accumulated ratings to fine-tune a reranking layer via DPO.

FEEDBACK_FILE = 'feedback_log.json'

def load_feedback():
    if os.path.exists(FEEDBACK_FILE):
        with open(FEEDBACK_FILE) as f:
            return json.load(f)
    return []

def save_feedback(log):
    with open(FEEDBACK_FILE, 'w') as f:
        json.dump(log, f, indent=2)

def rate_match(headline, client_id, relevant: bool, reason: str = ''):
    log    = load_feedback()
    result = match_headline(headline, client_id)
    entry  = {
        'timestamp':    datetime.now().isoformat(),
        'client':       CLIENTS[client_id]['name'],
        'headline':     headline,
        'relevant':     relevant,
        'reason':       reason,
        'system_score': result.get('score', 0),
        'system_tier':  result.get('tier', 'unknown'),
        'top_match':    result['results'][0]['topic'] if result['results'] else None,
    }
    log.append(entry)
    save_feedback(log)
    print(f"{'Relevant' if relevant else 'Not relevant'} - logged. (Total: {len(log)})")
    if reason:
        print(f'Reason: {reason}')

def rate_from_digest(client_id):
    matches = all_matches[client_id]
    strong = matches['strong']
    adj    = matches['adjacent']
    all_m  = sorted(strong, key=lambda x: x['score'], reverse=True) + \
             sorted(adj,    key=lambda x: x['score'], reverse=True)
    print(f"\nRating matches for {CLIENTS[client_id]['name']}")
    print("Commands: y=relevant, n=not relevant, s=skip, q=quit\n")
    for m in all_m:
        tier = 'STRONG' if m in strong else 'ADJACENT'
        print(f"[{tier}] {m['headline']}")
        print(f"Score: {m['score']:.3f}")
        verdict = input("Relevant? (y/n/s/q): ").strip().lower()
        if verdict == 'q':
            break
        if verdict == 's':
            continue
        reason = input("Reason (optional, enter to skip): ").strip()
        rate_match(m['headline'], client_id, relevant=(verdict == 'y'), reason=reason)
        print()
    print("Done! Run feedback_summary() to see results.")

def feedback_summary():
    log = load_feedback()
    if not log:
        print('No feedback logged yet.')
        return
    df_fb = pd.DataFrame(log)
    print(f'Total ratings: {len(df_fb)}')
    print(f"Precision (% relevant): {df_fb['relevant'].mean():.1%}")
    print('\nBy client:')
    print(df_fb.groupby('client')['relevant'].agg(['count','mean']).rename(columns={'mean':'precision'}))

print('Feedback system ready.')
print('Usage: rate_from_digest("client1")  - walk through all matches interactively')
print('       feedback_summary()        - see precision stats')

Feedback system ready.
Usage: rate_from_digest("client1")  - walk through all matches interactively
       feedback_summary()        - see precision stats


In [23]:
for client_id in CLIENTS:
    rate_from_digest(client_id)


Rating matches for Client 1
Commands: y=relevant, n=not relevant, s=skip, q=quit

[STRONG] Dillon Brooks DUI arrest spotlights Arizona’s marijuana impairment laws
Score: 0.711


KeyboardInterrupt: Interrupted by user